# 03 — Run the audit pipeline

Runs a claims batch through the whole pipeline on a Colab GPU (`src/pipeline.py`):
Isolation Forest + Autoencoder → SHAP → classifier (flagged claims) → ICD-11 reference
text → Llama 3.1 8B narrative (inconsistent verdicts only) → AuditReport rows in SQLite.

| Output (in Drive) | |
|---|---|
| `db/sha_fraud.sqlite` | pending reports the dashboard reads |
| `experiments/pipeline_runs/*.json` | run log: every stage per claim, plus evaluation against the labels |

**Before running:** T4 GPU runtime; a trained classifier in
`experiments/classifier/<version>/` (notebook 02); `data/processed/<version>/` and
`artifacts/icd11_cache.json` in Drive. The narrative model is the 4-bit
`hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4` (Llama 3.1 Community License),
served by vLLM on the same GPU; the first narrative waits a few minutes for it to load.

The default batch is the Deployment/Demonstration Pool (DR-08), never the test split.

In [2]:
! python3 --version

In [3]:
DATASET_VERSION = 'v0.6-20k'
BRANCH = 'dev'
BATCH = 'deployment_pool'
LIMIT = None        # e.g. 100 for a quick run
CLASSIFIER_RUN = '20261018T181622_seed7'   # a run_id folder name; None picks the run with the best validation F1
LLM_MODEL = 'hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4'
WRITE_PREDICTIONS = False   # True for evaluation runs ('validation', or 'test' with FINAL)
FINAL = False               # True only for the single held-out run on BATCH = 'test'

## 1. Bootstrap: Drive, repository, paths

In [4]:
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, sys
REPO_DIR = '/content/claimguard'

DRIVE_ENV = '/content/drive/MyDrive/claimguard/.env'

def load_env_file(path):
    # Minimal .env reader (KEY=value lines), so this cell needs no extra package.
    if not os.path.exists(path):
        return
    for line in open(path):
        line = line.strip()
        if line and not line.startswith('#') and '=' in line:
            key, value = line.split('=', 1)
            os.environ.setdefault(key.strip(), value.strip().strip('"\''))

def github_token():
    # The kernel runs on Colab's VM, so a local .env (VS Code's jupyter.envFile) never
    # reaches it. Read the .env kept in Drive, then a Colab secret, then prompt.
    load_env_file(DRIVE_ENV)
    if os.getenv('GITHUB_TOKEN'):
        return os.environ['GITHUB_TOKEN']
    try:
        from google.colab import userdata
        return userdata.get('GITHUB_TOKEN')
    except Exception:
        pass
    from getpass import getpass
    return getpass('GITHUB_TOKEN (read-only, not saved): ').strip()

if not os.path.exists(REPO_DIR):
    token = github_token()
    if not token:
        raise RuntimeError('No GITHUB_TOKEN: add a Colab secret or enter it at the prompt')
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH,
                    f'https://{token}@github.com/victorgpt0/claimguard.git', REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '-q'], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

COMMIT = subprocess.run(['git', 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
ROOT = '/content/drive/MyDrive/claimguard'
PROCESSED = f'{ROOT}/data/processed/{DATASET_VERSION}'
CACHE = f'{ROOT}/artifacts/icd11_cache.json'
EXPERIMENTS = f'{ROOT}/experiments/classifier/{DATASET_VERSION}'
# DR-08: only the Deployment/Demonstration Pool goes to the review database; other splits
# get their own evaluation database so held-out claims never reach the review queue.
DB = f'{ROOT}/db/sha_fraud.sqlite' if BATCH == 'deployment_pool' else f'{ROOT}/db/evaluation_{DATASET_VERSION}_{BATCH}.sqlite'
PREDICTIONS = f'{ROOT}/experiments/predictions/{DATASET_VERSION}'
if BATCH == 'test' and not FINAL:
    raise ValueError('The test split is read once, for the final evaluation: set FINAL = True deliberately')
RUN_LOGS = f'{ROOT}/experiments/pipeline_runs'
ENV_LOGS = f'{ROOT}/artifacts/environment_logs'
for path in (f'{PROCESSED}/{BATCH}.parquet', CACHE):
    if not os.path.exists(path):
        raise FileNotFoundError(f'{path} missing: upload it to Drive first')
for d in (os.path.dirname(DB), RUN_LOGS, ENV_LOGS):
    os.makedirs(d, exist_ok=True)
print('commit', COMMIT)

In [5]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!pip install -q vllm==0.19.0 peft shap
# Colab preinstalls torchao 0.10, which this project doesn't use; peft >= 0.21 refuses to
# start LoRA when an old torchao is present ("only versions above 0.16.0 are supported").
!pip uninstall -q -y torchao
import torch
print('cuda', torch.cuda.is_available())

## 2. Choose the classifier checkpoint

In [6]:
import json, glob
runs = {os.path.basename(os.path.dirname(p)): json.load(open(p)) for p in glob.glob(f'{EXPERIMENTS}/*/config.json')}
if not runs:
    raise FileNotFoundError(f'No trained classifier in {EXPERIMENTS}: run notebook 02 first')
run_id = CLASSIFIER_RUN or max(runs, key=lambda r: runs[r]['metrics']['f1'])
CLASSIFIER = f'{EXPERIMENTS}/{run_id}'
m = runs[run_id]['metrics']
print(f"{run_id}: validation precision {m['precision']:.3f}, recall {m['recall']:.3f}, f1 {m['f1']:.3f}")

## 3. Run the pipeline

In [7]:
limit = f'--limit {LIMIT}' if LIMIT else ''
# Stage-1 thresholds come from config.example.yaml (src.detection.tune_stage1). With
# --predictions the run is also written in the shared format for src.evaluation.heldout.
predictions = f'--predictions {PREDICTIONS}' if WRITE_PREDICTIONS else ''
final = '--final' if FINAL else ''
!python -m src.pipeline --data {PROCESSED} --batch {BATCH} {limit} --cache {CACHE} \
    --classifier {CLASSIFIER} --db {DB} --llm-provider vllm --llm-model {LLM_MODEL} --log-dir {RUN_LOGS} \
    --config {REPO_DIR}/config/config.example.yaml {predictions} {final}

## 4. Read some reports

Check the narratives yourself and record the review in `docs/ai_use_log.md`.

In [8]:
import sqlite3, pandas as pd
with sqlite3.connect(DB) as conn:
    reports = pd.read_sql('''SELECT r.id, c.icd11_code, c.clinical_notes, v.verdict, v.confidence,
                                    r.shap_attribution, r.llm_narrative
                             FROM AuditReport r JOIN Claim c ON c.id = r.claim_id
                             LEFT JOIN ClassifierVerdict v ON v.id = r.verdict_id
                             ORDER BY r.id DESC''', conn)
print(len(reports), 'reports;', reports['llm_narrative'].notna().sum(), 'with a narrative')
for row in reports[reports['llm_narrative'].notna()].head(5).itertuples():
    print(f'\n#{row.id} [{row.icd11_code}] {row.verdict} ({row.confidence:.2f})')
    print('NOTES:', row.clinical_notes)
    print('NARRATIVE:', row.llm_narrative)
import datetime
!pip freeze > {ENV_LOGS}/requirements_frozen_{datetime.date.today().isoformat()}_pipeline.txt